# Stability selection

Stability selection {cite}`meinshausen2010stability` is a subsampling
procedure that adds finite-sample false-positive control to boosting's
variable selection. Cross-validation for choosing $m_{\text{stop}}$ is in
[](model_selection.ipynb); [](../usage/case_studies) applies stability
selection to data.

Stability selection repeatedly fits boosting on half-samples and retains learners
selected often enough to exceed a frequency threshold. Under the assumptions
specified below, the threshold bounds the expected number of false selections.

## The procedure

1. Run boosting on many subsamples of size $\lfloor n/2 \rfloor$, stopping each as
   soon as $q$ distinct base learners have been selected.
2. For each learner $j$ and boosting iteration $m$, record the **selection
   frequency** $\hat\pi_j(m) = \frac{1}{B}\sum_b \mathbf{1}\{j \text{ selected
   within the first } m \text{ iterations in subsample } b\}$.
3. Return the **stable set** $\hat S = \{ j : \max_m \hat\pi_j(m) \ge \pi_{\text{thr}} \}$
   of learners whose frequency, maximized over iterations, reaches a
   threshold $\pi_{\text{thr}}$. Because $\hat\pi_j(m)$ cannot decrease in
   $m$, the maximum is the frequency at the end of each subsample's path.

A high frequency means the procedure repeatedly selected that learner across
subsamples. Correlated substitutes can split a useful effect across
several learners, while a stable proxy can be selected often. The threshold
defines the reported set; a learner in it can be a proxy for a predictor that
was not measured or not selected.

## Error control

Let $p$ be the number of candidate learners and $\mathrm{PFER}$ the expected
number of falsely selected learners (per-family error rate): an expected
count of false selections. The inequality relates $q$, $\pi_{\text{thr}}$,
and PFER. Under the original exchangeability assumption (the selection of
noise learners is exchangeable across noise learners, and the procedure does
no worse than random guessing),

$$
\mathrm{PFER} \le \frac{q^2}{(2\pi_{\text{thr}} - 1)\,p}.
$$

`stabsel` takes two of $(q, \pi_{\text{thr}}, \mathrm{PFER})$ and computes
the third from the bound: given $q$ and PFER, the cutoff is the smallest
$\pi_{\text{thr}}$ for which the bound is at most PFER; given $q$ and the
cutoff, PFER is the value of the bound. Sharper bounds hold with
**complementary-pairs** subsampling (draw $B$ pairs of disjoint half-samples,
the `"SS"` sampler) combined with an assumption on the shape of the
distribution of a noise learner's selection frequency: unimodal (a single
peak) or $r$-concave (a stronger shape condition)
{cite}`shah2013variable,hofner2015controlling`. `mboostrs` implements these bound
variants and matches R's `stabs` package.

### Choosing `q`, `PFER` and `cutoff`

The three parameters interact, so you typically fix two and derive the third:

| Parameter | What it controls | Typical value | How to set it |
|---|---|---|---|
| `q` | Number of distinct learners each subsample fit selects before it stops | 5–20 | About twice the number of true signals you expect, provided the derived cutoff is at most 1 |
| `PFER` | Expected number of false positives | 1–2 | Your tolerance for false discoveries |
| `cutoff` ($\pi_{\text{thr}}$) | Selection frequency threshold | 0.6–0.9 | Derived from $q$ and PFER; higher = more conservative |

**Practical recipe:**

1. Set `PFER = 1` (expect at most 1 false positive on average).
2. Set `q` to about twice the number of signals you believe are present
   (e.g., `q = 5` if you expect 2–3 true signals among 50 candidates; the
   worked example below gives the cutoff 0.75 for this setting).
3. Let `cutoff` be derived; under the plain bound
   $\pi_{\text{thr}} = \frac{1}{2}(1 + q^2 / (p \cdot \mathrm{PFER}))$.
4. If the derived `cutoff` is below 0.6, increase `q` or decrease `PFER`. If
   it exceeds 1, decrease `q`, increase `PFER`, or use `sampling_type="SS"`
   with a shape assumption.
5. Run with `B = 50–100` subsamples for stable frequency estimates.

```{warning}
`q` is the **number of distinct learners** that boosting may select in a
subsample before the subsample fit stops; it is a tuning value, separate from
the number of true signals. Setting `q` too small restricts the selection
path and may miss weak signals. Setting it too large raises the cutoff (the
PFER bound is quadratic in $q$) and can make the bound unattainable.
```

## Interpreting selection frequencies

The output of `stabsel` is:

- `selected`: the indices of learners that exceed `cutoff`.
- `max`: the per-learner maximum selection frequency across all iterations.
- `cutoff`: the threshold actually used (may differ from the input if derived).

Learners above `cutoff` enter the stable set. Frequency alone is not the
probability that a learner is a true effect: it also reflects `q`, learner
flexibility, correlated alternatives, sampling structure, and model error.

A selection frequency is the proportion of subsamples in which a learner was
selected; it has no p-value or confidence-interval interpretation, and it is
not generally monotone in effect size.

A **selection-probability path** plot shows each learner's frequency against
the cutoff $\pi_{\text{thr}}$, revealing which learners clear the threshold
and which alternatives compete. A large `q`
lets more learners enter each fit and weakens the error bound.

```{note}
The frequencies depend on `q` and `mstop`. Different `q` values change which
learners are considered per subsample, and `mstop` controls how many iterations
the subsample fit runs. The default `mstop` is often reasonable, but for very
high-dimensional problems with many weak signals, increasing `mstop` can help
weak signals be selected more consistently.
```

### Worked example: choosing PFER

Suppose $p = 50$ candidate learners and we set $q = 5$ and
$\mathrm{PFER} = 1$. The bound gives

$$
\pi_{\text{thr}} = \frac{1}{2}\Bigl(1 + \frac{q^2}{p \cdot \mathrm{PFER}}\Bigr)
               = \frac{1}{2}\Bigl(1 + \frac{25}{50}\Bigr) = \frac{1}{2}\cdot\frac{3}{2} = 0.75.
$$

Any learner selected in at least 75% of the $B$ subsamples is declared stable.
If instead $q = 10$ (with the same $p$ and PFER), the bound gives

$$
\pi_{\text{thr}} = \frac{1}{2}\Bigl(1 + \frac{100}{50}\Bigr) = 1.5,
$$

which exceeds 1 — impossible. The bound is vacuous for this combination: $q$
is too large relative to PFER. The remedy is to reduce $q$ back to 5, or
relax PFER (e.g. $\mathrm{PFER} = 2$ brings $q=10$ back to a boundary value
of exactly 1.0).

With complementary-pairs subsampling (`sampling_type="SS"`) and a unimodality
assumption on the selection distribution, the bound is tighter than the
plain formula (see {cite}`hofner2015controlling` for the exact tail bound).
For $q=10$, $p=50$, $\mathrm{PFER}=1$ (the combination that is
vacuous under the plain bound), the unimodal SS bound gives a valid
$\pi_{\text{thr}} \approx 0.88$. The `mboostrs` parameter solver gives the
same values:



In [1]:
from mboostrs.stabsel import _stabsel_parameters

# Non-unimodal (plain Meinshausen-Buhlmann) bound: q=5 gives a valid cutoff.
pars_valid = _stabsel_parameters(p=50, q=5, PFER=1, B=100,
                                  assumption="none", sampling_type="MB")
print("q=5,  PFER=1, plain bound:", pars_valid)
assert abs(pars_valid["cutoff"] - 0.75) < 1e-9

# The same bound at q=10 is vacuous: the raw threshold (1.5) exceeds 1, so the
# solver clamps to cutoff=1.0 and reports the PFER actually achieved there
# (which is larger than the PFER requested -- the guarantee fails).
pars_vacuous = _stabsel_parameters(p=50, q=10, PFER=1, B=100,
                                    assumption="none", sampling_type="MB")
print("q=10, PFER=1, plain bound:", pars_vacuous)
assert pars_vacuous["cutoff"] == 1.0
assert pars_vacuous["PFER"] > 1.0  # achieved PFER exceeds the requested budget

# Complementary-pairs subsampling with a unimodality assumption recovers a
# valid (< 1) cutoff for the same q=10, PFER=1 that was vacuous above.
pars_ss = _stabsel_parameters(p=50, q=10, PFER=1, B=100,
                               assumption="unimodal", sampling_type="SS")
print("q=10, PFER=1, unimodal + SS bound:", pars_ss)
assert pars_ss["cutoff"] < 1.0


q=5,  PFER=1, plain bound: {'cutoff': 0.75, 'q': 5, 'PFER': 1.0, 'p': 50, 'B': 100, 'sampling_type': 'MB', 'assumption': 'none'}
q=10, PFER=1, plain bound: {'cutoff': 1.0, 'q': 10, 'PFER': 2.0, 'p': 50, 'B': 100, 'sampling_type': 'MB', 'assumption': 'none'}
q=10, PFER=1, unimodal + SS bound: {'cutoff': 0.88, 'q': 10, 'PFER': 0.99, 'p': 50, 'B': 100, 'sampling_type': 'SS', 'assumption': 'unimodal'}


### Empirical check: the stable set against the true signal

The bound controls an *expectation* over repeated data draws, so a single
run cannot verify it directly. It can show that true predictors are
selected far more consistently across subsamples than pure-noise ones. The
data are $n = 300$ rows with 23 standard normal predictors, of which three
carry signal, $y = 2x_0 - 1.5x_1 + x_2 + \varepsilon$ with
$\varepsilon \sim N(0, 1)$. The model has a linear learner (`bols`) per
predictor and, as `glmboost` adds by default, an intercept learner, so there
are $p = 24$ candidate learners; $m_{\text{stop}} = 200$ and step length 0.1. Stability selection
uses $q = 6$, PFER = 1, complementary pairs (`"SS"`) with the unimodal
assumption, and $B = 50$.

In [2]:
import warnings
import numpy as np
import mboostrs as mb

warnings.filterwarnings("ignore")
rng = np.random.default_rng(0)
n = 300
n_noise = 20
n_signal = 3
X = rng.normal(size=(n, n_signal + n_noise))
beta = np.array([2.0, -1.5, 1.0])
y = X[:, :n_signal] @ beta + rng.normal(size=n)

cols = [f"x{i}" for i in range(X.shape[1])]
data = {c: X[:, i] for i, c in enumerate(cols)}
data["y"] = y
formula = "y ~ " + " + ".join(f"bols({c})" for c in cols)
fit = mb.glmboost(formula, data=data, family=mb.Gaussian(),
                   control=mb.boost_control(mstop=200, nu=0.1))

res = mb.stabsel(fit, q=6, PFER=1, sampling_type="SS", assumption="unimodal",
                  B=50, verbose=False)
names = np.array(res.learner_names)
selected_names = set(names[res.selected])
true_names = set(cols[:n_signal])
print(f"cutoff = {res.cutoff:.2f}")
print(f"selected: {sorted(selected_names)}")

signal_freq = res.max_[[list(names).index(c) for c in cols[:n_signal]]]
noise_freq = res.max_[[list(names).index(c) for c in cols[n_signal:]]]
print(f"true-signal max frequencies: {signal_freq}")
print(f"noise max frequency: mean={noise_freq.mean():.3f}  max={noise_freq.max():.3f}")

assert selected_names == true_names
assert signal_freq.min() > noise_freq.max()


cutoff = 0.84
selected: [np.str_('x0'), np.str_('x1'), np.str_('x2')]
true-signal max frequencies: [1. 1. 1.]
noise max frequency: mean=0.134  max=0.660


The derived cutoff is 0.84. The stable set is exactly the three true
predictors, each selected in every subsample (frequency 1.0). The noise
predictors have a mean maximum frequency of 0.134 and a largest value of
0.66, below the cutoff, although some of them are selected in some
subsamples.

## Complementary-pairs subsampling

The default **complementary-pairs** (SS) sampler
{cite}`shah2013variable,hofner2015controlling` draws $B$
pairs of disjoint half-samples: each observation is either in the subsample or
its complement, never both. This improves the bound because the selection
frequencies of noise variables are anti-correlated across the pair; combined
with a unimodality or $r$-concavity assumption on the selection distribution,
it can recover a usable threshold in regimes where the plain bound is vacuous
(see *Worked example: choosing PFER*).

Use `sampling_type="SS"` for the complementary-pairs bounds. Set
`sampling_type="MB"` explicitly for independent half-samples and the
original Meinshausen–Bühlmann bound.

## Stability selection and `cvrisk`

`cvrisk` chooses a stopping iteration by estimated held-out loss. Stability
selection retains learners whose subsample selection frequencies exceed a
threshold chosen with a false-positive budget. It uses frequencies along the
selection path rather than one prediction-optimal $m_{\text{stop}}$. A combined
workflow is:

1. Run `stabsel` to identify the stable set of predictors.
2. Fit a model using only those predictors.
3. Use `cvrisk` to choose `mstop` for the final model.

This two-step procedure decouples **model structure** (which variables) from
**model complexity** (how many iterations).

```{note}
Stability selection's guarantee is on the **expected number of false
selections**; it says nothing about power. Increasing `q` or `PFER` admits more
learners; lowering them is more conservative. The complementary-pairs sampler
(`sampling_type="SS"`, the default) gives cutoffs that depend on the shape
assumption: for the settings of *Empirical check: the stable set against the
true signal* ($p = 24$, $q = 6$, PFER = 1, $B = 50$) the cutoff is 1.0 with
`assumption="none"`, 0.84 with `"unimodal"` (the default) and 0.69 with
`"r-concave"`. The stronger the assumption, the lower the cutoff, and the
bound holds only if the assumption does.
```

## Post-fit selectors: `stabsel`, `deselect` and `probe`

`mb.stabsel` provides error-rate control under the assumptions of the chosen
bound. The other two post-fit selectors use heuristic rules. `mb.deselect`
(Strömer et al. 2022) drops learners whose share of the total risk reduction
falls below a threshold and refits. `mb.probe` (Thomas et al. 2017) adds a
permuted copy of every term and keeps learners selected before the first
permuted copy. Neither heuristic provides an error-rate guarantee.

False-positive control does not guarantee recovery of weak signals. A learner
never selected on any subsample has frequency zero and cannot enter the stable
set. Correlated learners can split selections, leaving each below the cutoff
even when one is selected in nearly every subsample.
`mb.substitution_sets(stab)` identifies groups with that pattern;
`mb.equivalence_sets(model)` lists pairs of learners whose function spaces
overlap by 0.9 or more, where the overlap is the largest fraction of one
learner's fitted functions that the other can reproduce (1 for an exact
copy). These reports help locate interchangeable predictors; they do not
identify a causal member of the group.

## Distributional models

For location, scale and shape (LSS) models fitted with `gamboostLSS`, stability
selection operates on the **combined** learner set
$\{\text{learner} \times \text{distribution parameter}\}$ using non-cyclic
updates (each iteration updates only the one distribution parameter whose
update most reduces the risk), so a covariate can be declared stable for the location but not the scale
(or vice versa). The error-control mathematics is unchanged; only the candidate
set is enlarged.

In practice, use `stabsel_glmboostLSS` or `stabsel_gamboostLSS` with the same
`q` and `PFER` settings as for single-predictor models. The output gives per
(parameter, learner) selection frequencies; a variable like `bbs(x)` can be
stable for $\mu$ but not for $\sigma$.

## Summary

- Stability selection complements CV-based $m_{\text{stop}}$ selection by
  controlling the expected number of false selections (PFER) during variable
  selection, rather than optimizing prediction error.
- The procedure subsamples the data $B$ times, fits the boosting path per
  subsample until $q$ distinct learners are selected, and records which
  learners were selected. Variables selected in at least $\pi_{\text{thr}}$ of
  subsamples are deemed stable; the plain PFER bound gives
  $\pi_{\text{thr}} = \frac{1}{2}(1 + q^2 / (p \cdot \mathrm{PFER}))$.
- Complementary-pairs subsampling (`sampling_type="SS"`), together with a
  unimodality or $r$-concavity assumption, gives a tighter bound than the
  plain formula above. For LSS models, each distribution parameter has its
  own selection frequency matrix; the PFER bound applies to the combined
  set of parameter-qualified candidate learners.

## See also

- [](model_selection.ipynb) — the complementary role of CV-based stopping.
- [](../usage/stability_selection) — runnable code examples.